In [ ]:
import json
import os

data_folder = "../data/t20s_male_json"
files = os.listdir(data_folder)
print(f"Number of match files found: {len(files)}")
print(f"First few files: {files[:5]}")

In [ ]:
sample_file = os.path.join(data_folder, files[0])
with open(sample_file) as f:
    match = json.load(f)

print(match.keys())

In [ ]:
print(match['info']['venue'])
print(match['info'].get('city', 'Not recorded'))
print(match['info']['dates'])
print(match['info']['teams'])
print(match['info']['outcome'])

In [ ]:
# Look at the structure of the innings data first
innings = match['innings']
print(f"Number of innings: {len(innings)}")
print(f"Keys in first innings: {innings[0].keys()}")
print(f"Batting team: {innings[0]['team']}")

In [ ]:
first_over = innings[0]['overs'][0]
print(first_over)

In [ ]:
import pandas as pd

rows = []
for inning in match['innings']:
    team = inning['team']
    for over in inning['overs']:
        over_num = over['over']
        for ball_num, delivery in enumerate(over['deliveries']):
            runs_info = delivery.get('runs', {})
            rows.append({
                'team': team,
                'over': over_num,
                'ball': ball_num + 1,
                'batter': delivery.get('batter'),
                'bowler': delivery.get('bowler'),
                'non_striker': delivery.get('non_striker'),
                'runs_batter': runs_info.get('batter', 0),
                'runs_extras': runs_info.get('extras', 0),
                'runs_total': runs_info.get('total', 0),
                'wicket': 'wickets' in delivery
            })

df = pd.DataFrame(rows)
print(f"Total balls in this match: {len(df)}")
df.head(10)

In [ ]:
df.groupby('team')['runs_total'].sum()

In [ ]:
import json
import os
import pandas as pd

data_folder = "../data/t20s_male_json"
all_files = [f for f in os.listdir(data_folder) if f.endswith('.json')]
print(f"Total match files to process: {len(all_files)}")

all_rows = []
errors = 0

for filename in all_files:
    filepath = os.path.join(data_folder, filename)
    try:
        with open(filepath) as f:
            match = json.load(f)

        match_id = filename.replace('.json', '')
        venue = match['info'].get('venue', 'Unknown')
        teams = match['info'].get('teams', [])
        outcome = match['info'].get('outcome', {})
        winner = outcome.get('winner', None)

        for inning in match['innings']:
            team = inning['team']
            for over in inning['overs']:
                over_num = over['over']
                for ball_num, delivery in enumerate(over['deliveries']):
                    runs_info = delivery.get('runs', {})
                    all_rows.append({
                        'match_id': match_id,
                        'venue': venue,
                        'team': team,
                        'winner': winner,
                        'over': over_num,
                        'ball': ball_num + 1,
                        'runs_total': runs_info.get('total', 0),
                        'wicket': 'wickets' in delivery
                    })
    except Exception as e:
        errors += 1
        continue

full_df = pd.DataFrame(all_rows)
print(f"Total balls across all matches: {len(full_df)}")
print(f"Files that failed to process: {errors}")
full_df.head()

In [ ]:
full_df.to_csv("../data/all_matches_ball_by_ball.csv", index=False)
print("Saved successfully")

In [ ]:
def build_situations(data_folder):
    situation_rows = []
    files = [f for f in os.listdir(data_folder) if f.endswith('.json')]

    for filename in files:
        filepath = os.path.join(data_folder, filename)
        try:
            with open(filepath) as f:
                match = json.load(f)

            if len(match['innings']) < 2:
                continue  # skip matches with no second innings (e.g. abandoned)

            outcome = match['info'].get('outcome', {})
            winner = outcome.get('winner', None)
            if winner is None:
                continue  # skip ties/no-results, keep it simple for now

            first_innings = match['innings'][0]
            second_innings = match['innings'][1]
            chasing_team = second_innings['team']

            # Calculate the target: total runs scored in first innings + 1
            target = 0
            for over in first_innings['overs']:
                for delivery in over['deliveries']:
                    target += delivery.get('runs', {}).get('total', 0)
            target += 1

            # Walk through second innings ball by ball
            current_score = 0
            wickets_fallen = 0
            balls_bowled = 0
            total_balls_in_innings = 120  # T20 = 20 overs = 120 balls

            for over in second_innings['overs']:
                for delivery in over['deliveries']:
                    balls_bowled += 1
                    runs = delivery.get('runs', {}).get('total', 0)
                    current_score += runs
                    if 'wickets' in delivery:
                        wickets_fallen += len(delivery['wickets'])

                    runs_needed = target - current_score
                    balls_remaining = total_balls_in_innings - balls_bowled
                    wickets_in_hand = 10 - wickets_fallen

                    situation_rows.append({
                        'match_id': filename.replace('.json', ''),
                        'chasing_team': chasing_team,
                        'runs_needed': runs_needed,
                        'balls_remaining': balls_remaining,
                        'wickets_in_hand': wickets_in_hand,
                        'current_run_rate': (current_score / balls_bowled) * 6 if balls_bowled > 0 else 0,
                        'required_run_rate': (runs_needed / balls_remaining) * 6 if balls_remaining > 0 else 0,
                        'chasing_team_won': 1 if winner == chasing_team else 0
                    })
        except Exception:
            continue

    return pd.DataFrame(situation_rows)

situations_df = build_situations(data_folder)
print(f"Total situations (balls in 2nd innings): {len(situations_df)}")
situations_df.head(10)

In [ ]:
situations_df.to_csv("../data/situations.csv", index=False)
print("Saved successfully")

In [ ]:
import pandas as pd

situations_df = pd.read_csv("../data/situations.csv")
print(f"Loaded {len(situations_df)} situations")
situations_df.head()